In [ ]:
# ── COLAB CELL: GPT-4o + o4-mini Baseline Evaluation ────────────────────────
#
# Run this on Colab. Set your OpenAI API key below.
# Benchmark file should be on Drive: NIPS_2026/MedConflict-Bench/medconflict_bench_v2.jsonl
#
# Estimated cost: ~$8-12 for both models on 1,261 examples
# Estimated time: ~40-60 min total (rate limited)

!pip install openai scikit-learn tqdm -q

import os
import json
import time
import numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from google.colab import userdata
from openai import OpenAI

# ============================================================================
# CONFIG — SET YOUR API KEY HERE
# ============================================================================

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

BENCH_PATH = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/medconflict_bench_v2.jsonl")
OUTPUT_DIR = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/eval_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    {"name": "o4-mini", "model_id": "o4-mini"},
]

DELAY = 2.0  # seconds between API calls (rate limiting)
MAX_PER_STRATUM = None  # Set to 50 for quick test, None for full eval

# ============================================================================
# PROMPT
# ============================================================================

SYSTEM_PROMPT = """You are a clinical pharmacist reviewing a patient's medication order.
Your task is to determine if there is a clinically significant conflict between the
prescribed medication and the patient's clinical data.

A conflict can be:
1. A drug prescribed to a patient with a documented allergy or contraindicated condition
2. A drug prescribed when a lab value exceeds a safety threshold
3. A drug-drug interaction that becomes dangerous given the patient's lab values

Respond with EXACTLY this JSON format (no other text):
{
  "conflict_detected": true/false,
  "severity": "none" | "moderate" | "major",
  "conflict_type": "none" | "drug_allergy" | "drug_disease" | "numerical_threshold" | "drug_drug_interaction",
  "explanation": "Brief clinical reasoning (1-2 sentences)",
  "recommended_action": "none" | "contraindicated" | "dose_reduction" | "hold_dose" | "monitor_closely" | "discontinue"
}"""

def build_prompt(example):
    parts = ["=== MEDICATION ORDER REVIEW ===\n"]
    drug = example.get("drug", "Unknown")
    dose = example.get("dose_val", "")
    unit = example.get("dose_unit", "")
    route = example.get("route", "")
    drug_line = f"Prescribed Medication: {drug}"
    if dose and unit and str(dose) != "nan" and str(unit) != "nan":
        drug_line += f" {dose} {unit}"
    if route and str(route) != "nan":
        drug_line += f" ({route})"
    parts.append(drug_line)
    if example.get("interacting_drug"):
        parts.append(f"Co-prescribed Medication: {example['interacting_drug']}")
    lab_name = example.get("lab_name")
    lab_value = example.get("lab_value")
    lab_unit = example.get("lab_unit")
    if lab_name and lab_value is not None:
        parts.append(f"\nRelevant Lab Result:")
        lab_line = f"  {lab_name}: {lab_value}"
        if lab_unit and str(lab_unit) != "nan":
            lab_line += f" {lab_unit}"
        parts.append(lab_line)
    condition = example.get("conflict_condition", "")
    if example.get("stratum") == 1 and condition and condition != "none":
        parts.append(f"\nPatient History / Active Diagnoses:")
        parts.append(f"  - {condition}")
    parts.append("\n=== END OF ORDER ===")
    parts.append("\nIs there a clinically significant conflict with this medication order?")
    parts.append("Respond with the JSON format specified.")
    return "\n".join(parts)

# ============================================================================
# API CALL
# ============================================================================

def call_openai(client, model, example):
    prompt = build_prompt(example)
    is_o_series = model.startswith("o")
    params = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
    }
    if is_o_series:
        params["max_completion_tokens"] = 4096
    else:
        params["max_tokens"] = 512
        params["temperature"] = 0

    for attempt in range(3):
        try:
            response = client.chat.completions.create(**params)
            raw = response.choices[0].message.content
            try:
                # Clean potential markdown wrapping
                clean_raw = raw.strip()
                if clean_raw.startswith("```"):
                    clean_raw = clean_raw.split("\n", 1)[-1].rsplit("\n", 1)[0].strip()
                if clean_raw.lower().startswith("json"):
                    clean_raw = clean_raw[4:].strip()
                parsed = json.loads(clean_raw)
                return parsed, raw
            except:
                return {"conflict_detected": False, "error": "JSON_PARSE_ERROR"}, raw
        except Exception as e:
            print(f"    Retry {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    return {"conflict_detected": False, "error": "API_ERROR"}, "ERROR"

# ============================================================================
# METRICS
# ============================================================================

def compute_metrics(results, model_name):
    y_true = np.array([1 if r["gt_conflict"] else 0 for r in results])
    y_pred = np.array([1 if r["pred_conflict"] else 0 for r in results])
    overall = {
        "model": model_name,
        "n": len(results),
        "accuracy": round((y_true == y_pred).mean(), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    overall.update({"tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)})
    per_stratum = {}
    for s in [0, 1, 2, 3]:
        mask = np.array([r["stratum"] == s for r in results])
        if mask.sum() == 0: continue
        st, sp = y_true[mask], y_pred[mask]
        m = {"n": int(mask.sum()), "accuracy": round((st == sp).mean(), 4)}
        if s == 0:
            m["specificity"] = round(1 - sp.mean(), 4)
            m["false_positive_rate"] = round(sp.mean(), 4)
        else:
            m["detection_rate"] = round(sp.mean(), 4)
            m["recall"] = round(recall_score(st, sp, zero_division=0), 4)
            m["f1"] = round(f1_score(st, sp, zero_division=0), 4)
        per_stratum[s] = m
    cliff = {}
    if 1 in per_stratum and 2 in per_stratum:
        s1 = per_stratum[1].get("detection_rate", 0)
        s2 = per_stratum[2].get("detection_rate", 0)
        cliff = {"absolute_drop": round(s1 - s2, 4), "relative_drop_pct": round((s1 - s2) / max(s1, 0.001) * 100, 1)}
    return {"overall": overall, "per_stratum": per_stratum, "cliff": cliff}

# ============================================================================
# MAIN EVALUATION
# ============================================================================

print("=" * 70)
print("MedConflict-Bench v2 — GPT-4o & o4-mini Evaluation")
print("=" * 70)

examples = [json.loads(l) for l in open(BENCH_PATH, encoding="utf-8")]
test_examples = [e for e in examples if e.get("split") == "test"]
print(f"\nTest examples: {len(test_examples)}")

client = OpenAI()
all_metrics = {}

for model_config in MODELS:
    model_name = model_config["name"]
    model_id = model_config["model_id"]
    ckpt_path = OUTPUT_DIR / f"{model_name}_results.jsonl"

    if ckpt_path.exists():
        results = [json.loads(l) for l in open(ckpt_path, encoding="utf-8")]
        done_ids = set(r["example_id"] for r in results)
        remaining = [e for e in test_examples if e["example_id"] not in done_ids]
    else:
        results = []
        remaining = test_examples

    print(f"\nEvaluating: {model_name} — {len(remaining)} remaining")

    for i, ex in enumerate(tqdm(remaining, desc=f"  {model_name}")):
        parsed, raw = call_openai(client, model_id, ex)
        results.append({
            "example_id": ex["example_id"],
            "stratum": ex["stratum"],
            "gt_conflict": ex.get("conflict_present", False),
            "pred_conflict": parsed.get("conflict_detected", False),
            "pred_severity": parsed.get("severity", "none"),
            "pred_type": parsed.get("conflict_type", "none"),
            "pred_action": parsed.get("recommended_action", "none"),
            "pred_explanation": parsed.get("explanation", ""),
            "raw_response": raw[:500],
            "rule_id": ex.get("rule_id", ""),
        })
        if (i + 1) % 50 == 0:
            with open(ckpt_path, "w", encoding="utf-8") as f:
                for r in results: f.write(json.dumps(r) + "\n")
        time.sleep(DELAY)

    with open(ckpt_path, "w", encoding="utf-8") as f:
        for r in results: f.write(json.dumps(r) + "\n")

    metrics = compute_metrics(results, model_name)
    all_metrics[model_name] = metrics

print("\n✅ EVALUATION COMPLETE")

In [ ]:
import os
bad_file = "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/eval_results/o4-mini_results.jsonl"
if os.path.exists(bad_file):
    os.remove(bad_file)
    print("Deleted bad o4-mini checkpoint")

# Also delete metrics
bad_metrics = "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/eval_results/o4-mini_metrics.json"
if os.path.exists(bad_metrics):
    os.remove(bad_metrics)
    print("Deleted bad o4-mini metrics")

In [ ]:
import json
from pathlib import Path

# IMPORTANT: Make sure this path matches where you saved the files in your Drive!
RESULTS_DIR = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/eval_results")

JSONL_FILE = RESULTS_DIR / "o4-mini_results.jsonl"
METRICS_FILE = RESULTS_DIR / "o4-mini_metrics.json"

# --- 1. PRINT SUMMARY METRICS ---
print("=" * 50)
print("🏆 o4-mini FINAL METRICS SUMMARY 🏆")
print("=" * 50)

if METRICS_FILE.exists():
    with open(METRICS_FILE, "r") as f:
        metrics = json.load(f)
        overall = metrics.get("overall", {})

    print(f"Accuracy:  {overall.get('accuracy')}")
    print(f"F1 Score:  {overall.get('f1')}")
    print(f"Precision: {overall.get('precision')}")
    print(f"Recall:    {overall.get('recall')}")
    print(f"TP: {overall.get('tp')} | FP: {overall.get('fp')}")
    print(f"TN: {overall.get('tn')} | FN: {overall.get('fn')}")
else:
    print(f"❌ Metrics file not found at: {METRICS_FILE}")

# --- 2. PRINT TOP 5 ERRORS ---
print("\n" + "=" * 50)
print("🔍 EXAMINING ERRORS 🔍")
print("=" * 50)

if JSONL_FILE.exists():
    error_count = 0
    with open(JSONL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)

            gt = data.get("gt_conflict")
            pred = data.get("pred_conflict")

            # If the model guessed wrong, print it out
            if gt != pred:
                error_count += 1
                print(f"Example ID: {data.get('example_id')} (Stratum: {data.get('stratum')})")
                print(f"  Truth: {gt} | Model Guessed: {pred}")
                print(f"  Raw Response:\n{data.get('raw_response')[:250]}...\n") # Trimmed to keep it clean

                if error_count >= 5: # Change this number to see more errors
                    break

    if error_count == 0:
        print("No errors found! Perfect score.")
else:
    print(f"❌ Results file not found at: {JSONL_FILE}")

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

RESULTS_DIR = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/eval_results")
JSONL_FILE = RESULTS_DIR / "o4-mini_results.jsonl"
METRICS_FILE = RESULTS_DIR / "o4-mini_metrics.json"

def compute_metrics(results, model_name):
    y_true = np.array([1 if r["gt_conflict"] else 0 for r in results])
    y_pred = np.array([1 if r["pred_conflict"] else 0 for r in results])

    overall = {
        "model": model_name,
        "n": len(results),
        "accuracy": round((y_true == y_pred).mean(), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    overall.update({"tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)})

    per_stratum = {}
    for s in [0, 1, 2, 3]:
        mask = np.array([r.get("stratum", -1) == s for r in results])
        if mask.sum() == 0: continue
        st, sp = y_true[mask], y_pred[mask]
        m = {"n": int(mask.sum()), "accuracy": round((st == sp).mean(), 4)}
        if s == 0:
            m["specificity"] = round(1 - sp.mean(), 4)
            m["false_positive_rate"] = round(sp.mean(), 4)
        else:
            m["detection_rate"] = round(sp.mean(), 4)
            m["recall"] = round(recall_score(st, sp, zero_division=0), 4)
            m["f1"] = round(f1_score(st, sp, zero_division=0), 4)
        per_stratum[s] = m

    cliff = {}
    if 1 in per_stratum and 2 in per_stratum:
        s1 = per_stratum[1].get("detection_rate", 0)
        s2 = per_stratum[2].get("detection_rate", 0)
        cliff = {"absolute_drop": round(s1 - s2, 4), "relative_drop_pct": round((s1 - s2) / max(s1, 0.001) * 100, 1)}

    return {"overall": overall, "per_stratum": per_stratum, "cliff": cliff}

# 1. Load the data
print("Loading results...")
results = [json.loads(l) for l in open(JSONL_FILE, encoding="utf-8")]

# 2. Compute metrics
print("Calculating metrics...")
metrics = compute_metrics(results, "o4-mini")

# 3. Save the missing JSON file
with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# 4. Print the final summary table
print("\n" + "=" * 80)
print("RESULTS — Per-Stratum Detection Rates (THE PERFORMANCE CLIFF)")
print("=" * 80)

ps = metrics["per_stratum"]
cliff = metrics["cliff"]

s0 = f"{ps.get(0, {}).get('specificity', '—'):.3f}"
s1 = f"{ps.get(1, {}).get('detection_rate', '—'):.3f}"
s2 = f"{ps.get(2, {}).get('detection_rate', '—'):.3f}"
s3 = f"{ps.get(3, {}).get('detection_rate', '—'):.3f}"
drop = f"{cliff.get('absolute_drop', 0):+.3f}"
f1 = f"{metrics['overall']['f1']:.3f}"

print(f"{'Model':<15} {'S0-Spec':<10} {'S1-Cat':<10} {'S2-Num':<10} {'S3-MH':<10} {'Cliff':<10} {'Overall F1'}")
print("-" * 80)
print(f"{'o4-mini':<15} {s0:<10} {s1:<10} {s2:<10} {s3:<10} {drop:<10} {f1}")
print("\n✅ Metrics successfully saved to your Google Drive!")

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

RESULTS_DIR = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/eval_results")

# Pointing to the GPT-4o files
JSONL_FILE = RESULTS_DIR / "gpt-4o_results.jsonl"
METRICS_FILE = RESULTS_DIR / "gpt-4o_metrics.json"

def compute_metrics(results, model_name):
    y_true = np.array([1 if r["gt_conflict"] else 0 for r in results])
    y_pred = np.array([1 if r["pred_conflict"] else 0 for r in results])

    overall = {
        "model": model_name,
        "n": len(results),
        "accuracy": round((y_true == y_pred).mean(), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    overall.update({"tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)})

    per_stratum = {}
    for s in [0, 1, 2, 3]:
        mask = np.array([r.get("stratum", -1) == s for r in results])
        if mask.sum() == 0: continue
        st, sp = y_true[mask], y_pred[mask]
        m = {"n": int(mask.sum()), "accuracy": round((st == sp).mean(), 4)}
        if s == 0:
            m["specificity"] = round(1 - sp.mean(), 4)
            m["false_positive_rate"] = round(sp.mean(), 4)
        else:
            m["detection_rate"] = round(sp.mean(), 4)
            m["recall"] = round(recall_score(st, sp, zero_division=0), 4)
            m["f1"] = round(f1_score(st, sp, zero_division=0), 4)
        per_stratum[s] = m

    cliff = {}
    if 1 in per_stratum and 2 in per_stratum:
        s1 = per_stratum[1].get("detection_rate", 0)
        s2 = per_stratum[2].get("detection_rate", 0)
        cliff = {"absolute_drop": round(s1 - s2, 4), "relative_drop_pct": round((s1 - s2) / max(s1, 0.001) * 100, 1)}

    return {"overall": overall, "per_stratum": per_stratum, "cliff": cliff}

# 1. Load the data
print("Loading GPT-4o results...")
if not JSONL_FILE.exists():
    print(f"❌ Could not find {JSONL_FILE}")
else:
    results = [json.loads(l) for l in open(JSONL_FILE, encoding="utf-8")]

    # 2. Compute metrics
    print("Calculating metrics...")
    metrics = compute_metrics(results, "gpt-4o")

    # 3. Save the JSON file
    with open(METRICS_FILE, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    # 4. Print the final summary table
    print("\n" + "=" * 80)
    print("RESULTS — Per-Stratum Detection Rates (THE PERFORMANCE CLIFF)")
    print("=" * 80)

    ps = metrics["per_stratum"]
    cliff = metrics["cliff"]

    s0 = f"{ps.get(0, {}).get('specificity', '—'):.3f}" if 0 in ps else "—"
    s1 = f"{ps.get(1, {}).get('detection_rate', '—'):.3f}" if 1 in ps else "—"
    s2 = f"{ps.get(2, {}).get('detection_rate', '—'):.3f}" if 2 in ps else "—"
    s3 = f"{ps.get(3, {}).get('detection_rate', '—'):.3f}" if 3 in ps else "—"
    drop = f"{cliff.get('absolute_drop', 0):+.3f}" if cliff else "—"
    f1 = f"{metrics['overall']['f1']:.3f}"

    print(f"{'Model':<15} {'S0-Spec':<10} {'S1-Cat':<10} {'S2-Num':<10} {'S3-MH':<10} {'Cliff':<10} {'Overall F1'}")
    print("-" * 80)
    print(f"{'gpt-4o':<15} {s0:<10} {s1:<10} {s2:<10} {s3:<10} {drop:<10} {f1}")
    print("\n✅ GPT-4o metrics successfully calculated and saved!")

RESULT




In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# IMPORTANT: Path to your results
RESULTS_DIR = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/eval_results")

MODELS = ["o4-mini", "gpt-4o"]

def compute_metrics_unrounded(results, model_name):
    y_true = np.array([1 if r["gt_conflict"] else 0 for r in results])
    y_pred = np.array([1 if r["pred_conflict"] else 0 for r in results])

    # Using float() to ensure it serializes nicely to JSON, but NO rounding!
    overall = {
        "model": model_name,
        "n": len(results),
        "accuracy": float((y_true == y_pred).mean()),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    overall.update({"tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)})

    per_stratum = {}
    for s in [0, 1, 2, 3]:
        mask = np.array([r.get("stratum", -1) == s for r in results])
        if mask.sum() == 0: continue
        st, sp = y_true[mask], y_pred[mask]

        m = {"n": int(mask.sum()), "accuracy": float((st == sp).mean())}
        if s == 0:
            m["specificity"] = float(1 - sp.mean())
            m["false_positive_rate"] = float(sp.mean())
        else:
            m["detection_rate"] = float(sp.mean())
            m["recall"] = float(recall_score(st, sp, zero_division=0))
            m["f1"] = float(f1_score(st, sp, zero_division=0))
        per_stratum[s] = m

    cliff = {}
    if 1 in per_stratum and 2 in per_stratum:
        s1 = per_stratum[1].get("detection_rate", 0.0)
        s2 = per_stratum[2].get("detection_rate", 0.0)
        cliff = {
            "absolute_drop": float(s1 - s2),
            "relative_drop_pct": float((s1 - s2) / max(s1, 0.0000000001) * 100)
        }

    return {"overall": overall, "per_stratum": per_stratum, "cliff": cliff}

# Run for both models
for model_name in MODELS:
    print(f"\n{'='*80}")
    print(f"PROCESSING RAW METRICS FOR: {model_name}")
    print(f"{'='*80}")

    jsonl_file = RESULTS_DIR / f"{model_name}_results.jsonl"
    metrics_file = RESULTS_DIR / f"{model_name}_metrics_unrounded.json" # Saving as a new file so we don't overwrite your clean ones

    if not jsonl_file.exists():
        print(f"❌ Could not find {jsonl_file}. Skipping.")
        continue

    print(f"Loading {model_name} results...")
    results = [json.loads(l) for l in open(jsonl_file, encoding="utf-8")]

    print("Calculating raw, unrounded metrics...")
    metrics = compute_metrics_unrounded(results, model_name)

    # Save the exact numbers
    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    # Print out the raw numbers
    ps = metrics["per_stratum"]
    cliff = metrics["cliff"]

    print(f"\n--- UNROUNDED RESULTS ({model_name}) ---")
    print(f"Overall F1:  {metrics['overall']['f1']}")
    print(f"S0-Spec:     {ps.get(0, {}).get('specificity', '—')}")
    print(f"S1-Cat:      {ps.get(1, {}).get('detection_rate', '—')}")
    print(f"S2-Num:      {ps.get(2, {}).get('detection_rate', '—')}")
    print(f"S3-MH:       {ps.get(3, {}).get('detection_rate', '—')}")

    if cliff:
        print(f"Cliff Drop:  {cliff.get('absolute_drop')}")
        print(f"Cliff %:     {cliff.get('relative_drop_pct')}%")

    print(f"✅ Raw metrics saved to: {metrics_file.name}")